# Filename Parsing — RAVDESS Metadata

Parse RAVDESS filenames into structured metadata and perform basic validation.

In [ ]:
# Setup path and imports
from pathlib import Path
import sys

# Resolve project root in common notebook working directories
cwd = Path.cwd().resolve()
candidates = [cwd, cwd / "SER_Project", cwd.parent, cwd.parent / "SER_Project"]
PROJECT_ROOT = next((p for p in candidates if (p / "pipeline" / "data_utils.py").exists()), None)

if PROJECT_ROOT is None:
    raise RuntimeError(f"Could not locate SER_Project/pipeline/data_utils.py from {cwd}")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Import from pipeline package
from pipeline.data_utils import parse_ravdess_filename, build_ravdess_metadata

In [13]:
# Discover all WAV files under data directory
# Guard against PROJECT_ROOT being None
if PROJECT_ROOT is None:
	possible = [cwd, cwd.parent, Path.cwd()]
	found = next((p for p in possible if (p / "data").exists()), None)
	if found is None:
		raise RuntimeError("Could not locate 'data' directory. Ensure PROJECT_ROOT is set or run the setup cell.")
	data_root = found / "data"
else:
	data_root = PROJECT_ROOT / "data"

wav_files = sorted(data_root.rglob("*.wav"))

print(f"Found {len(wav_files)} audio files")
wav_files[:3]

Found 2452 audio files


[WindowsPath('C:/Users/wblut/Documents/My Projects/Python/CSE 432/Project_CSE432-532/SER_Project/data/Actor_01/03-01-01-01-01-01-01.wav'),
 WindowsPath('C:/Users/wblut/Documents/My Projects/Python/CSE 432/Project_CSE432-532/SER_Project/data/Actor_01/03-01-01-01-01-02-01.wav'),
 WindowsPath('C:/Users/wblut/Documents/My Projects/Python/CSE 432/Project_CSE432-532/SER_Project/data/Actor_01/03-01-01-01-02-01-01.wav')]

In [10]:
# Test parsing on a single filename
sample_record = parse_ravdess_filename(wav_files[0])
sample_record

{'filepath': 'C:\\Users\\wblut\\Documents\\My Projects\\Python\\CSE 432\\Project_CSE432-532\\SER_Project\\data\\Actor_01\\03-01-01-01-01-01-01.wav',
 'filename': '03-01-01-01-01-01-01.wav',
 'modality_code': 3,
 'vocal_channel_code': 1,
 'emotion_code': 1,
 'intensity_code': 1,
 'statement_code': 1,
 'repetition': 1,
 'actor_id': 1,
 'modality': 'audio-only',
 'vocal_channel': 'speech',
 'emotion': 'neutral',
 'intensity': 'normal',
 'statement': 'Kids are talking by the door'}

In [11]:
# Build metadata table from all files
meta_df = build_ravdess_metadata(wav_files)
meta_df.head()

,filepath,filename,modality_code,vocal_channel_code,emotion_code,intensity_code,statement_code,repetition,actor_id,modality,vocal_channel,emotion,intensity,statement
0,C:\Users\wblut\Documents\My Projects\Python\CS...,03-01-01-01-01-01-01.wav,3,1,1,1,1,1,1,audio-only,speech,neutral,normal,Kids are talking by the door
1,C:\Users\wblut\Documents\My Projects\Python\CS...,03-01-01-01-01-02-01.wav,3,1,1,1,1,2,1,audio-only,speech,neutral,normal,Kids are talking by the door
2,C:\Users\wblut\Documents\My Projects\Python\CS...,03-01-01-01-02-01-01.wav,3,1,1,1,2,1,1,audio-only,speech,neutral,normal,Dogs are sitting by the door
3,C:\Users\wblut\Documents\My Projects\Python\CS...,03-01-01-01-02-02-01.wav,3,1,1,1,2,2,1,audio-only,speech,neutral,normal,Dogs are sitting by the door
4,C:\Users\wblut\Documents\My Projects\Python\CS...,03-01-02-01-01-01-01.wav,3,1,2,1,1,1,1,audio-only,speech,calm,normal,Kids are talking by the door


In [5]:
# Basic validation checks
print("Rows:", len(meta_df))
print("\nMissing values by column:")
display(meta_df.isna().sum())

print("\nEmotion distribution:")
display(meta_df["emotion"].value_counts().sort_index())

print("\nVocal channel distribution:")
display(meta_df["vocal_channel"].value_counts().sort_index())

Rows: 2452

Missing values by column:


filepath              0
filename              0
modality_code         0
vocal_channel_code    0
emotion_code          0
intensity_code        0
statement_code        0
repetition            0
actor_id              0
modality              0
vocal_channel         0
emotion               0
intensity             0
statement             0
dtype: int64


Emotion distribution:


emotion
angry        376
calm         376
disgust      192
fearful      376
happy        376
neutral      188
sad          376
surprised    192
Name: count, dtype: int64


Vocal channel distribution:


vocal_channel
song      1012
speech    1440
Name: count, dtype: int64

In [6]:
# Export metadata for use in later notebooks
output_path = data_root / "metadata.csv"
meta_df.to_csv(output_path, index=False)
print(f"Saved metadata to: {output_path}")

Saved metadata to: C:\Users\wblut\Documents\My Projects\Python\CSE 432\Project_CSE432-532\SER_Project\data\metadata.csv
